In [5]:
# --- BLOQUE 0: INFRAESTRUCTURA MÍNIMA ---
import sys
import gzip
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from scipy.optimize import curve_fit
from astropy.cosmology import Planck18 as cosmo

print("[SYSTEM] Iniciando Protocolo FTP DIRECTO (Modo Depuración)...")

# --- BLOQUE 1: DESCARGA ---
FTP_URL = "https://heasarc.gsfc.nasa.gov/FTP/heasarc/dbase/dump/heasarc_fermilgrb.tdat.gz"
print(f"[NET] Atacando URL: {FTP_URL}")

try:
    response = requests.get(FTP_URL)
    response.raise_for_status()
    print(f"[SUCCESS] Archivo descargado ({len(response.content)/1024:.1f} KB).")

    with gzip.open(io.BytesIO(response.content), 'rt', encoding='utf-8') as f:
        content = f.read()

except Exception as e:
    print(f"[CRITICAL] Error de descarga: {e}")
    sys.exit()

# --- BLOQUE 2: DIAGNÓSTICO VISUAL (CRUCIAL) ---
print("-" * 40)
print("[DEBUG] PRIMERAS 20 LÍNEAS DEL ARCHIVO:")
print("-" * 40)
lines = content.split('\n')
for i, line in enumerate(lines[:20]):
    print(f"L{i}: {line.strip()}")
print("-" * 40)

# --- BLOQUE 3: PARSER INTELIGENTE ---
print("[PROC] Buscando cabecera de datos...")

column_names = []
data_start_row = None # This will be the first row of actual data

# Step 1: Collect all column names from 'field[name] = ...' lines
# and simultaneously find the <DATA> marker
import re # Import regex for advanced parsing

for i, line in enumerate(lines):
    l_low = line.lower()

    # Try to extract column names from lines like 'field[name] = char16 ...'
    if l_low.startswith('field['):
        match = re.search(r'field\[(.*?)\]\s*=', line)
        if match:
            col_name = match.group(1).strip()
            column_names.append(col_name)

    # Find the actual start of the data block, typically marked by <DATA>
    if '<DATA>' in lines[i]:
        data_start_row = i + 1 # Data starts *after* the <DATA> marker
        print(f"[INFO] Marcador de inicio de datos '<DATA>' encontrado en línea {i}. Datos comienzan en línea {data_start_row}.")
        # We can break here if we assume all field definitions are *before* <DATA>
        # For robustness, we might continue if field definitions could appear after, but unlikely for TDAT.
        break

if not column_names:
    print("[CRITICAL] No se pudo extraer nombres de columna de las líneas 'field[name] = ...'.")
    sys.exit()
else:
    print(f"[INFO] Nombres de columna extraídos ({len(column_names)}): {column_names[:5]}...")

if data_start_row is None:
    print("[CRITICAL] No se encontró el marcador '<DATA>' para el inicio de los datos.")
    sys.exit()


# --- BLOQUE 4: CARGA DATAFRAME ---
try:
    buf = io.StringIO(content)
    # Skip rows until the actual data starts. The header will be set explicitly later.
    # The actual data lines are pipe-separated.
    df = pd.read_csv(buf, sep='|', skiprows=data_start_row, header=None, on_bad_lines='skip', engine='python')

    # Limpiar posibles filas vacías que puedan aparecer si el archivo tiene líneas extra al final
    df = df.dropna(how='all')

    # Assign column names after reading
    # Ensure the number of column names matches the DataFrame's columns
    if len(column_names) == len(df.columns):
        df.columns = [c.strip().upper() for c in column_names]
        print(f"[SUCCESS] DataFrame creado con columnas extraídas: {len(df.columns)}")
    elif len(df.columns) == len(column_names) + 1: # Common case: one extra column due to trailing delimiter
        print(f"[WARN] DataFrame tiene una columna extra ({len(df.columns)}) comparado con las extraídas ({len(column_names)}).")
        print("[WARN] Eliminando la última columna (posiblemente vacía).")
        df = df.iloc[:, :-1] # Drop only the last column
        df.columns = [c.strip().upper() for c in column_names]
        print(f"[SUCCESS] DataFrame ajustado y columnas asignadas: {len(df.columns)}")
    elif len(df.columns) == len(column_names) + 2: # Another common case: empty first column (due to initial '|') and empty last column (trailing '|')
        print(f"[WARN] DataFrame tiene dos columnas extra ({len(df.columns)}) comparado con las extraídas ({len(column_names)}).")
        print("[WARN] Eliminando la primera y la última columna (posiblemente vacías debido a delimitadores iniciales/finales)...")
        df = df.iloc[:, 1:-1] # Drop the first and last columns
        df.columns = [c.strip().upper() for c in column_names]
        print(f"[SUCCESS] DataFrame ajustado y columnas asignadas: {len(df.columns)}")
    else:
        print(f"[WARN] Número de columnas extraídas ({len(column_names)}) no coincide con el DataFrame ({len(df.columns)}).")
        print(f"[WARN] El DataFrame contiene {len(df.columns)} columnas. Posibles columnas detectadas: {list(df.columns)}")
        print("[WARN] No se pudo ajustar automáticamente las columnas, se usarán nombres genéricos.")
        df.columns = [f'COL_{i}' for i in range(len(df.columns))] # Assign generic names for now if mismatch


except Exception as e:
    print(f"[ERROR] Fallo al leer CSV: {e}")
    sys.exit()

# --- BLOQUE 5: MAPEO Y FÍSICA ---
def find_col(df, keywords):
    for col in df.columns:
        for k in keywords:
            # Some TDAT columns might have special chars or be part of a larger string
            if k.upper() in col.upper(): return col
    return None

col_z = find_col(df, ['REDSHIFT', 'Z'])
# En el dump TDAT, a veces energy se llama 'energy' o 'flux' si es GBM, pero en fermilgrb debería ser 'highest_energy'
col_e = find_col(df, ['HIGHEST_ENERGY', 'MAX_ENE', 'EXT_EMISSION_MAX', 'EXT_EMISSION_MAX_ENE'])
# A veces el TDAT de HEASARC tiene nombres truncados, ej: 'highest_ene'
col_t = find_col(df, ['HIGHEST_ENERGY_TIME', 'TIME_MAX', 'SCAT_TIME', 'T90', 'EXT_EMISSION_MAX_ENE_T'])
col_n = find_col(df, ['NAME', 'GRB'])

print(f"[MAP] Z: {col_z} | E: {col_e} | T: {col_t}")

if not col_z:
    print("[ERROR] No encuentro columna de Redshift.")
    sys.exit()

# Limpieza de datos numéricos (los TDAT usan cadenas raras para NULL)
def clean_numeric(series):
    return pd.to_numeric(series, errors='coerce')

df['Z_CLEAN'] = clean_numeric(df[col_z])

# Si no encontramos Energía/Tiempo perfectos, improvisamos con lo que haya para ver si corre
if not col_e:
    print("[WARN] No encuentro energía explícita. Usando 1 GeV dummy para test de tiempos.")
    df['E_CLEAN'] = 1.0
else:
    df['E_CLEAN'] = clean_numeric(df[col_e]) / 1000.0 # MeV -> GeV

if not col_t:
    # Si no hay tiempo de alta energía, usamos T90 o Trigger Time como referencia cero
    # Pero necesitamos un "retraso".
    print("[WARN] No encuentro tiempo de alta energía. Usando T90 como proxy de duración.")
    col_t_alt = find_col(df, ['T90', 'DURATION'])
    if col_t_alt:
        df['T_CLEAN'] = clean_numeric(df[col_t_alt])
    else:
        print("[CRITICAL] Sin datos de tiempo.")
        sys.exit()
else:
    df['T_CLEAN'] = clean_numeric(df[col_t])

# Filtros
mask = (df['Z_CLEAN'] > 0) & (df['E_CLEAN'] > 0) & (df['T_CLEAN'].notna())
final_data = df[mask]

clean_z = final_data['Z_CLEAN'].values
clean_e_gev = final_data['E_CLEAN'].values
clean_t = final_data['T_CLEAN'].values
clean_names = final_data[col_n].values if col_n else np.full(len(final_data), 'GRB') # Fallback if name col not found

print(f"[DATA] Eventos finales: {len(clean_z)}")

# --- BLOQUE 6: ANÁLISIS ---
print("[PHYS] Calculando Time-of-Flight...")

def get_k(z):
    z_grid = np.linspace(0, z, 50)
    Ez = np.sqrt(cosmo.Om0*(1+z_grid)**3 + cosmo.Ode0)
    integrand = (1+z_grid) / Ez
    return np.trapz(integrand, z_grid)

# Initialize ylabel with a default value
y_label_text = 'Valor Desconocido'

if len(clean_z) > 0:
    K_factors = np.array([get_k(z) for z in clean_z])
    # Lag normalizado (s/GeV) o Lag puro (s) si E es dummy
    if col_e:
        lags = clean_t / clean_e_gev
        y_label_text = 'Retraso Normalizado (s/GeV)'
    else:
        lags = clean_t
        y_label_text = 'Duración/Retraso (s)'
else:
    K_factors, lags = np.array([]), np.array([])

# Regresión
def model(k, m, c): return m * k + c

m_fit = 0
# Only attempt curve_fit if there are enough data points
if len(K_factors) > 2:
    try:
        # Ensure non-zero standard deviation for fitting
        if np.std(K_factors) > 0 and np.std(lags) > 0:
            popt, pcov = curve_fit(model, K_factors, lags)
            m_fit = popt[0]
        else:
            print("[WARN] Datos insuficientes para la regresión: poca variabilidad en K_factors o lags.")
    except RuntimeError as e:
        print(f"[WARN] No se pudo realizar la regresión de curva: {e}")
    except Exception as e:
        print(f"[WARN] Error inesperado en la regresión: {e}")

# Visualización
plt.figure(figsize=(12, 7))
plt.style.use('dark_background')

# Only attempt to plot if there is data
if len(K_factors) > 0:
    sc = plt.scatter(K_factors, lags, c=clean_e_gev, cmap='cool', s=80, alpha=0.8, label='HEASARC FTP')
    plt.colorbar(sc, label='Energía (GeV)')

    if m_fit != 0:
        # Ensure x range is valid for plotting
        if len(K_factors) > 0 and max(K_factors) > 0:
            x = np.linspace(0, max(K_factors)*1.1, 10)
            plt.plot(x, model(x, m_fit, popt[1]), color='magenta', linestyle='--', label=f'Fit (m={m_fit:.2f})')
        else:
            print("[WARN] K_factors no tiene rango válido para la línea de ajuste.")

    plt.axhline(0, color='white', linestyle=':')

    # Outliers: only annotate if there are enough points for sorting and names
    if len(lags) >= 3 and len(clean_names) == len(lags):
        for i in np.argsort(lags)[-3:]:
            plt.annotate(f"{clean_names[i]}", (K_factors[i], lags[i]), color='yellow', xytext=(5,5), textcoords='offset points')
    elif len(lags) > 0: # If less than 3, annotate all available points
        for i in range(len(lags)):
            if len(clean_names) == len(lags): # Only if names are available
                plt.annotate(f"{clean_names[i]}", (K_factors[i], lags[i]), color='yellow', xytext=(5,5), textcoords='offset points')
            else:
                plt.annotate(f"Point {i}", (K_factors[i], lags[i]), color='yellow', xytext=(5,5), textcoords='offset points')

plt.xlabel('Factor K (Distancia)')
plt.ylabel(y_label_text) # Use the correctly initialized variable
plt.title(f'TEST FTP (Debug Mode): Violación de Lorentz\nArchivo: {FTP_URL.split("/")[-1]}', color='white')

# Only show legend if there are items with labels
handles, labels = plt.gca().get_legend_handles_labels()
if handles:
    plt.legend()
else:
    print("[WARN] No hay elementos con etiquetas para mostrar en la leyenda.")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n[RESULT] Pendiente LIV: {m_fit:.4f}")

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



[SYSTEM] Iniciando Protocolo FTP DIRECTO (Modo Depuración)...
[NET] Atacando URL: https://heasarc.gsfc.nasa.gov/FTP/heasarc/dbase/dump/heasarc_fermilgrb.tdat.gz
[SUCCESS] Archivo descargado (67.1 KB).
----------------------------------------
[DEBUG] PRIMERAS 20 LÍNEAS DEL ARCHIVO:
----------------------------------------
L0: <HEADER>
L1: #
L2: #         TABLE: heasarc_fermilgrb
L3: #      LOCATION: dbms1.gsfc.nasa.gov:heasarcprod
L4: #    TOTAL ROWS: 231
L5: #
L6: # CREATION DATE: 2022-06-22 16:49:38
L7: # LAST MODIFIED: 2025-10-23 19:58:31
L8: #   EXGEST DATE: 2025-10-23 20:51:35
L9: #
L10: table_name = heasarc_fermilgrb
L11: table_description = "Fermi LAT Second Gamma-Ray Burst Catalog"
L12: table_document_url = http://heasarc.gsfc.nasa.gov/W3Browse/fermi/fermilgrb.html
L13: table_security = public
L14: #
L15: # Table Parameters
L16: #
L17: field[name] = char16  [meta.id;meta.main] (index) // Designation of the GRB in yymmddfff Format
L18: field[gcn_name] = char16  [meta.id] (index) 

TypeError: object of type 'NoneType' has no len()